# 07 · Categorical (Multi-class) and Binned (Histogram) Forecasts

The forecast is a probability vector $\mathbf p_t=(p_{t,1},\dots,p_{t,K})$ over $K$ categories, with $\sum_kp_{t,k}=1$.

* **Categorical / ordinal**: "Below normal 20% · Near normal 50% · Above normal 30%" (seasonal climate outlooks),
  demand class Low / Medium / High, credit-rating migration.
* **Binned density (histogram) forecast**: the categories are **intervals of a continuous variable**, e.g. the
  Survey of Professional Forecasters' GDP-growth histograms, or the probability of sales falling in each 2%-growth bin.
  Modern models do this too: **Chronos** quantises values into 4 096 bins and predicts a categorical distribution
  over them with a language model. Forecasting becomes classification.

## Theory

### Nominal vs ordinal
For **ordinal** classes, distance matters: forecasting "High" when "Medium" happens is less wrong than when "Low"
happens. The score should reward probability mass that is *close* to the outcome.

### Scores
| Score | Formula | Sensitive to distance? |
|---|---|---|
| Multi-category Brier | $\sum_k (p_k-o_k)^2$ | no |
| Log score / ignorance | $-\log p_{k^*}$ ($k^*$ = observed class); in bits: $-\log_2$ | no (local) |
| **Ranked Probability Score** | $\sum_{k=1}^{K-1}(P_k-O_k)^2$ with cumulative $P_k=\sum_{j\le k}p_j$ | **yes** |

RPS is the discrete analogue of CRPS: as bins get narrower, $\text{RPS}\times\text{bin width}\to\text{CRPS}$.

### Building categorical forecasts
1. **Discretise a continuous predictive distribution**: $p_k=\hat F(b_k)-\hat F(b_{k-1})$
2. **Direct classifier**: multinomial logit / ordinal logit / softmax networks
3. **Count ensemble members** per category (weather ensembles)

### Calibration of binned forecasts
Use a **randomised PIT**: $u=\hat F(b_{k^*-1})+V\cdot p_{k^*}$ with $V\sim U(0,1)$. It is uniform if the forecasts are calibrated.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(7)

## 1. Why distance matters: Brier vs RPS on a toy example
Observed class = "Low" (index 0) out of Low / Normal / High.

In [2]:
toy = pd.DataFrame({"Low": [0.5, 0.5], "Normal": [0.5, 0.0], "High": [0.0, 0.5]}, index=["A: near miss", "B: far miss"])
toy["Brier"] = [np.sum((toy.iloc[i, :3].to_numpy() - np.array([1, 0, 0])) ** 2) for i in range(2)]
toy["RPS"] = pf.rps(np.array([0, 0]), toy.iloc[:, :3].to_numpy())
toy

,Low,Normal,High,Brier,RPS
A: near miss,0.5,0.5,0.0,0.5,0.25
B: far miss,0.5,0.0,0.5,0.5,0.50


Both forecasts put 50% on the right class, so the Brier score is equal. RPS correctly prefers A, which put the remaining
mass on the *adjacent* category.

## 2. Monthly setup: one-step forecasts of YoY growth for four stores
Same design as nb 06: SARIMA on log-sales fitted on 2015–2019, one-step forecasts for 2020–2024 (240 forecasts).
The predictive distribution of YoY log-growth $G_t=\log y_t-\log y_{t-12}$ is $N(\mu_t-\log y_{t-12},\ \sigma_t^2)$.

In [3]:
retail = pf.make_retail_hierarchy()
CUT = 60
frames, clim_growth = [], []
for s in pf.BOTTOM_NAMES:
    ly = np.log(retail[s]); g = ly - ly.shift(12)
    fit = SARIMAX(ly.iloc[:CUT], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    pr = SARIMAX(ly, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).filter(fit.params).get_prediction(start=CUT)
    frames.append(pd.DataFrame({
        "store": s,
        "gm": np.asarray(pr.predicted_mean) - ly.shift(12).iloc[CUT:].to_numpy(),   # predictive mean of growth
        "gs": np.sqrt(np.asarray(pr.var_pred_mean)),
        "g": g.iloc[CUT:].to_numpy(),
        "g1": g.shift(1).iloc[CUT:].to_numpy(), "g2": g.shift(2).iloc[CUT:].to_numpy()}, index=ly.index[CUT:]))
    clim_growth.append(g.iloc[12:CUT])
df = pd.concat(frames)
clim_growth = pd.concat(clim_growth)
print(f"{len(df)} forecasts")

240 forecasts


## 3. Tercile forecasts: Below / Near / Above normal
"Normal" is defined from the training climatology: the 33rd and 67th percentiles of historical YoY growth.
By construction climatology says 1/3 each.

In [4]:
e1, e2 = np.quantile(clim_growth, [1 / 3, 2 / 3])
print(f"tercile edges: {e1:+.3f}, {e2:+.3f} (log growth)")
CLASSES = ["Below", "Near", "Above"]
df["cat"] = np.digitize(df.g, [e1, e2])              # 0, 1, 2

def tercile_probs(m, s):
    c1, c2 = stats.norm.cdf(e1, m, s), stats.norm.cdf(e2, m, s)
    return np.column_stack([c1, c2 - c1, 1 - c2])

P = {"Model (SARIMA)": tercile_probs(df.gm, df.gs),
     "Over-confident (σ/3)": tercile_probs(df.gm, df.gs / 3),
     "Climatology": np.full((len(df), 3), 1 / 3)}

# direct multinomial logit on lagged growth, trained on pre-2020 data
tr = []
for s in pf.BOTTOM_NAMES:
    ly = np.log(retail[s]); g = ly - ly.shift(12)
    tr.append(pd.DataFrame({"g1": g.shift(1), "g2": g.shift(2), "cat": np.digitize(g, [e1, e2])}).iloc[14:CUT])
tr = pd.concat(tr)
mnl = sm.MNLogit(tr.cat, sm.add_constant(tr[["g1", "g2"]])).fit(disp=0)
P["Multinomial logit"] = np.asarray(mnl.predict(sm.add_constant(df[["g1", "g2"]])))

obs = df.cat.to_numpy()
onehot = np.eye(3)[obs]
res = {}
for name, p in P.items():
    res[name] = {"RPS": pf.rps(obs, p).mean(), "multi-Brier": np.mean(np.sum((p - onehot) ** 2, axis=1)),
                 "ignorance (bits)": np.mean(-np.log2(np.clip(p[np.arange(len(obs)), obs], 1e-6, 1)))}
res = pd.DataFrame(res).T
res["RPSS vs clim"] = 1 - res.RPS / res.loc["Climatology", "RPS"]
res

tercile edges: -0.009, +0.103 (log growth)


,RPS,multi-Brier,ignorance (bits),RPSS vs clim
Model (SARIMA),0.282,0.525,1.213,0.304
Over-confident (σ/3),0.332,0.629,2.061,0.180
Climatology,0.406,0.667,1.585,0.000
Multinomial logit,0.360,0.613,1.453,0.111


In [5]:
FORECASTERS = ["Model (SARIMA)", "Over-confident (σ/3)", "Multinomial logit"]
fig = pf.subplots(1, 3, titles=[f"Reliability for class '{cls}'" for cls in CLASSES], shared_yaxes=True, height=420)
for k, cls in enumerate(CLASSES):
    for j, name in enumerate(FORECASTERS):
        pf.add_reliability(fig, obs == k, P[name][:, k], n_bins=6, name=name, color=pf.SERIES[j], row=1, col=k + 1,
                           diagonal=j == 0, showlegend=k == 0)
fig.show()

In [6]:
# what the forecasts look like over time for one store: stacked probabilities
s = "South_B"; m = (df.store == s).to_numpy()
p = P["Model (SARIMA)"][m]
ORDINAL = {"Below": pf.SERIES[0], "Near": pf.INK["neutral"], "Above": pf.SERIES[7]}   # diverging: cool / neutral / warm
fig = go.Figure()
for k, cls in enumerate(CLASSES):
    fig.add_trace(go.Scatter(x=df.index[m], y=p[:, k], mode="lines", stackgroup="p", name=f"P({cls})",
                             line=dict(width=0.5, color=pf.INK["surface"]), fillcolor=ORDINAL[cls],
                             hovertemplate=f"P({cls}) = %{{y:.2f}}<extra></extra>"))
fig.add_trace(go.Scatter(x=df.index[m], y=np.where(obs[m] == 0, 0.05, np.where(obs[m] == 1, 0.5, 0.95)), mode="markers",
                         name="observed class", marker=dict(color=pf.INK["primary"], size=8, line=dict(color=pf.INK["surface"], width=2)),
                         customdata=np.array(CLASSES)[obs[m]], hovertemplate="observed: %{customdata}<extra></extra>"))
fig.update_layout(title=f"{s}: tercile probability forecasts (stacked)", height=380, hovermode="x unified", yaxis_range=[0, 1],
                  yaxis_title="probability")
fig.show()

## 4. Binned (histogram) density forecasts
Ten growth bins: below −4%, −4…−2%, … , above +12%. Each forecast is a histogram. We compare the model, the
over-confident model, and a **climatological histogram** (historical bin frequencies, Laplace-smoothed).

In [7]:
edges = np.log1p(np.array([-0.04, -0.02, 0.0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12]))
full_edges = np.r_[-np.inf, edges, np.inf]
K = len(full_edges) - 1
labels = ["<-4%"] + [f"{a:+.0%}…{b:+.0%}" for a, b in zip(np.expm1(edges[:-1]), np.expm1(edges[1:]))] + [">+12%"]

def bin_probs(m, s):
    c = stats.norm.cdf(full_edges[None, :], np.asarray(m)[:, None], np.asarray(s)[:, None])
    return np.diff(c, axis=1)

hist_clim = (np.histogram(clim_growth, bins=full_edges)[0] + 1) / (len(clim_growth) + K)
B = {"Model (SARIMA)": bin_probs(df.gm, df.gs), "Over-confident (σ/3)": bin_probs(df.gm, df.gs / 3),
     "Climatology histogram": np.tile(hist_clim, (len(df), 1))}
bin_obs = np.digitize(df.g, edges)

res_b = {}
for name, p in B.items():
    p_obs = np.clip(p[np.arange(len(df)), bin_obs], 1e-6, 1)
    res_b[name] = {"RPS": pf.rps(bin_obs, p).mean(), "ignorance (bits)": np.mean(-np.log2(p_obs))}
pd.DataFrame(res_b).T

,RPS,ignorance (bits)
Model (SARIMA),1.378,2.770
Over-confident (σ/3),1.601,4.444
Climatology histogram,1.948,3.161


In [8]:
i = np.where(df.store == "North_A")[0][-1]
fig = pf.subplots(1, 2, titles=[f"Histogram forecast for North_A, {df.index[i]:%b %Y}", "Randomised PIT of binned forecasts"],
                  height=440)
fig.add_trace(go.Bar(x=labels, y=B["Model (SARIMA)"][i], name="model", marker_color=pf.SERIES[0],
                     hovertemplate="model: P(%{x}) = %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=labels, y=hist_clim, name="climatology", marker_color=pf.SERIES[1],
                     hovertemplate="climatology: P(%{x}) = %{y:.3f}<extra></extra>"), row=1, col=1)
top = max(B["Model (SARIMA)"][i].max(), hist_clim.max())
pf.add_vline(fig, labels[bin_obs[i]], 0, top, "observed bin", color=pf.INK["primary"], dash="dot", row=1, col=1, width=2)
fig.update_layout(barmode="group")

# randomised PIT for binned forecasts (same colour per forecaster as the bars: model blue, climatology orange)
BIN_COLOR = {"Model (SARIMA)": pf.SERIES[0], "Climatology histogram": pf.SERIES[1], "Over-confident (σ/3)": pf.SERIES[2]}
for name, p in B.items():
    cum = np.cumsum(p, axis=1)
    cdf_lo = np.where(bin_obs > 0, cum[np.arange(len(df)), np.maximum(bin_obs - 1, 0)], 0.0)   # F(lower bin edge)
    u = cdf_lo + rng.uniform(size=len(df)) * p[np.arange(len(df)), bin_obs]
    pf.add_step_hist(fig, u, bins=10, range_=(0, 1), name=f"PIT: {name}", color=BIN_COLOR[name], row=1, col=2)
fig.add_hline(y=1, line=dict(color=pf.INK["primary"], dash="dash", width=1), row=1, col=2)
fig.update_xaxes(tickangle=-45, row=1, col=1)
fig.update_xaxes(title_text="PIT value", row=1, col=2)
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.3, x=0), margin=dict(b=140))
fig.show()

## 5. Takeaways
* Categorical forecasts are discretised CDFs. Derive them from a distributional model when possible, so that every
  category definition stays coherent.
* Use **RPS** for ordinal categories and bins. Use the log score / ignorance when every category matters equally
  and you want to punish over-confidence.
* Report a **skill score vs climatology** (RPSS). Raw RPS values depend on the number of categories.
* Binned forecasts lose information inside each bin, so choose bin widths that match the decision.

### Exercises
1. Increase to 30 bins and check that `RPS × bin width` approaches CRPS (nb 00) for the model forecasts.
2. Fit an *ordinal* logit (`statsmodels.miscmodels.ordinal_model.OrderedModel`) and compare it with the multinomial logit.
3. Build tercile forecasts by counting members of the SARIMA ensemble (nb 05) instead of using the Gaussian CDF.